# 03 · Axe 3 Assistants et agents de code (Edwin DREAN-AVRILLON)

**Questions :** quels outils, lesquels garde-t-on, lesquels progressent ? Ce notebook retrace aussi **un piège de dénominateur** qui a changé notre calcul.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # le notebook vit dans notebooks/ : on importe devia depuis la racine
import pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 160, "display.max_columns", 30)
from devia import data, population
d26 = data.prepare_2026(data.load_2026()); d25 = data.prepare_2025(data.load_2025())
p26, p25 = d26[d26.is_pro_dev], d25[d25.is_pro_dev]          # population par défaut : développeurs professionnels
print(f"2026 : {len(p26):,} développeurs professionnels sur {len(d26):,} répondants | 2025 : {len(p25):,} sur {len(d25):,}")
from devia.axes import agents

2026 : 20,220 développeurs professionnels sur 30,903 répondants | 2025 : 34,146 sur 49,191


In [2]:
agents.tool_table(p26).head(8)[["tool","share_used","share_want","admiration","k_used","k_used_both"]].round(3)

,tool,share_used,share_want,admiration,k_used,k_used_both
0,Claude Code,0.713,0.673,0.768,6098,3741
1,GitHub Copilot,0.612,0.296,0.436,5236,3258
2,OpenAI Codex,0.309,0.266,0.550,2645,1663
3,Cursor,0.273,0.176,0.352,2337,1475
4,Google Antigravity,0.152,0.119,0.363,1303,848
5,JetBrains AI,0.137,0.090,0.331,1175,738
6,Open Code,0.137,0.181,0.656,1174,825
7,Gemini Code Assist,0.132,0.090,0.325,1131,726


## Le piège : à qui rapporter l'« admiration » ?
Les colonnes « utilisé » et « souhaité » n'ont pas le même nombre de répondants. Une case « souhaité » vide peut vouloir dire « aucun » ou « pas de réponse ».

In [3]:
H,W,A = "AICodeAgentHaveWorkedWith","AICodeAgentWantToUseNext6Months","AICodeAgentAdmired"
has = lambda s, t: s.fillna("").str.split(";").apply(lambda x: t in x)
users = p26[p26[H].notna() & has(p26[H], "Claude Code")]
print("utilisateurs de Claude Code :", len(users), "| dont ont répondu à « souhaité » :", users[W].notna().sum())
print("admiration parmi TOUS les utilisateurs      :", round(has(users[A], "Claude Code").mean(), 3))
print("admiration parmi ceux qui ont répondu aux 2  :", round(has(users[users[W].notna()][A], "Claude Code").mean(), 3))

utilisateurs de Claude Code : 6098 | dont ont répondu à « souhaité » : 3741
admiration parmi TOUS les utilisateurs      : 0.471
admiration parmi ceux qui ont répondu aux 2  : 0.768


**Décision :** on rapporte l'admiration aux utilisateurs qui ont répondu aux deux questions. Compter une absence de réponse comme un « non » aurait fait passer Claude Code de 77 % à 47 %.

In [4]:
agents.providers(p26, 6).round(3)

,option,n,k,share,lo,hi
0,Anthropic,8347,6625,0.794,0.785,0.802
1,OpenAI,8347,6334,0.759,0.750,0.768
2,Google AI,8347,4435,0.531,0.521,0.542
3,Microsoft,8347,2204,0.264,0.255,0.274
4,DeepSeek,8347,1454,0.174,0.166,0.182
5,Alibaba Qwen,8347,831,0.100,0.093,0.106


In [5]:
agents.tools_per_person(p26).round(3)

,category,n,k,share,lo,hi
0,1,8549,2131,0.249,0.240,0.259
1,2,8549,2263,0.265,0.255,0.274
2,3,8549,1703,0.199,0.191,0.208
3,4 et +,8549,2452,0.287,0.277,0.296
